# Exploratory Data Analysis (EDA)

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Handling Missing Values](#handling-missing-values)
3. [Feature Distributions](#feature-distributions)
4. [Possible Biases](#possible-biases)
5. [Correlations](#correlations)


## Imports

In [2]:
import os
import polars as pl
import ipregistry as ipr
import matplotlib.pyplot as plt
import seaborn as sns

from common.downloader import RipeAtlasDownloader, DataType

## Dataset Download

In [3]:
start = "2026-05-18T0000"
end = "2026-05-18T0100"

radl = RipeAtlasDownloader(cache_dir="../ripe-cache")
ping_paths = list(radl.download_range(DataType.PING, start, end))
traceroute_paths = list(radl.download_range(DataType.TRACEROUTE, start, end))

Cache hit: ../ripe-cache/2026-05-18/ping-2026-05-18T0000.parquet
Cache hit: ../ripe-cache/2026-05-18/traceroute-2026-05-18T0000.parquet


. [Correlations](#correlations)


## Dataset Overview

[Provide a high-level overview of the dataset. This should include the source of the dataset, the number of samples, the number of features, and example showing the structure of the dataset.]


### Ping data

Here's a breakdown of every field in the schema:

**Measurement metadata**
- **`msm_id`** — unique ID of the measurement on RIPE Atlas
- **`msm_name`** — human-readable name of the measurement type (e.g. `"Ping"`)
- **`group_id`** — if the measurement is part of a group/batch, the shared group ID
- **`type`** — measurement type (e.g. `"ping"`, `"traceroute"`, `"dns"`)
- **`step`** — interval in seconds between repeated measurements (for periodic ones)
- **`timestamp`** — Unix timestamp of when the measurement was conducted

**Probe info**
- **`prb_id`** — unique ID of the RIPE Atlas probe that ran the measurement
- **`from`** — the probe's public-facing IP address (how it's seen on the internet)
- **`src_addr`** — source IP in the outgoing packet header (may differ from `from` if behind NAT)
- **`fw`** — firmware version of the probe
- **`mver`** — measurement software/protocol version

**Destination**
- **`dst_name`** — the target as originally specified (hostname or IP string)
- **`dst_addr`** — the resolved IP address that was actually pinged
- **`af`** — address family: `4` for IPv4, `6` for IPv6

**Ping results**
- **`result`** — list of individual ping replies, each a struct with:
  - **`rtt`** — round-trip time in milliseconds for that ping
  - **`x`** — `"*"` if the ping timed out / no reply
  - **`error`** — error string if something went wrong (e.g. `"N"` for network unreachable)
- **`sent`** — number of ping packets sent
- **`rcvd`** — number of replies received
- **`dup`** — number of duplicate replies received
- **`min`** — minimum RTT across all replies (ms)
- **`max`** — maximum RTT across all replies (ms)
- **`avg`** — average RTT across all replies (ms)
- **`lts`** — last time synced — seconds since the probe last synced its clock (low = reliable timestamps)

**Packet properties**
- **`proto`** — protocol used: `"ICMP"` or `"UDP"`
- **`size`** — size of the ping packet payload in bytes
- **`ttl`** — TTL (Time To Live) value of the received reply packet

### Traceroute Data

Here's the typical RIPE Atlas traceroute schema with all fields explained:

**Measurement metadata** *(same as ping)*
- **`msm_id`** — unique ID of the measurement
- **`msm_name`** — measurement type name (e.g. `"Traceroute"`)
- **`group_id`** — group/batch ID if part of a series
- **`type`** — `"traceroute"`
- **`step`** — interval in seconds for periodic measurements
- **`timestamp`** — Unix timestamp of when the measurement ran

**Probe info** *(same as ping)*
- **`prb_id`** — probe ID
- **`from`** — probe's public-facing IP
- **`src_addr`** — source IP in the packet header
- **`fw`** — probe firmware version
- **`mver`** — measurement software version

**Destination** *(same as ping)*
- **`dst_name`** — target as originally specified
- **`dst_addr`** — resolved destination IP
- **`af`** — address family: `4` or `6`

**Traceroute-specific: `result`**
This is the key difference from ping. It's a **list of hops**, each containing:
- **`hop`** — hop number (1, 2, 3, ...)
- **`result`** — list of probe replies at that hop (usually 3), each with:
  - **`from`** — IP address of the router that replied at this hop
  - **`rtt`** — round-trip time to this hop in milliseconds
  - **`ttl`** — TTL value in the reply packet
  - **`size`** — size of the reply packet
  - **`x`** — `"*"` if no reply (hop didn't respond)
  - **`err`** — ICMP error type if applicable (e.g. `"N"` for network unreachable)
  - **`late`** — integer flag if the reply arrived late (out of order)
  - **`dup`** — flag if this was a duplicate reply

**Packet properties**
- **`proto`** — protocol used: `"ICMP"`, `"UDP"`, or `"TCP"`
- **`size`** — packet size in bytes
- **`paris_id`** — Paris traceroute flow ID — used to keep packets on the same network path (avoids load balancer shuffling); `0` means classic traceroute
- **`lts`** — seconds since last clock sync
- **`ttl`** — TTL of the final reply (at destination)
- **`dup`** — duplicate replies at the overall measurement level
- **`endtime`** — Unix timestamp when the traceroute completed (traceroutes take time, unlike a single ping)

The biggest structural difference from ping is the **nested two-level `result`**: hops → per-hop replies, which lets you reconstruct the full path through the network.

## Data Filtering

### Ping

In [4]:
ping_df = pl.scan_parquet(ping_paths).filter(
    pl.col("af") == 4,  # limit experiments to only IPv4
    pl.col("proto") == "ICMP",  # just to be sure...
    pl.col("rcvd") > 0,  # only keep measurements that actually received a reply
    pl.col("lts") < 3600,  # only keep measurements that synced their clock within the last hour
    pl.col("size") == pl.scan_parquet(ping_paths).select(pl.col("size").mode()).collect().item(),
    # only keep measurements with the same size
).select([
    "timestamp",
    "from",
    "dst_addr",
    "sent",
    "rcvd",
    "dup",
    "min",
    "max",
    "avg",
]).rename({
    "from": "src_addr",
}).unique(
    ["src_addr", "dst_addr"]
).with_columns(
    timestamp=pl.from_epoch(pl.col("timestamp"), time_unit="s")
)

ping_df.head().collect()

timestamp,src_addr,dst_addr,sent,rcvd,dup,min,max,avg
datetime[μs],str,str,i64,i64,i64,f64,f64,f64
2026-05-18 00:37:07,"""79.98.72.65""","""195.158.60.23""",3,3,0,9.087709,9.542209,9.335084
2026-05-18 00:17:08,"""103.40.209.7""","""170.39.224.221""",3,3,0,165.299707,165.410973,165.348514
2026-05-18 00:44:08,"""185.157.208.248""","""217.243.179.165""",3,3,0,131.502399,131.759728,131.613003
2026-05-18 00:15:16,"""202.124.238.42""","""62.50.171.162""",3,3,0,296.226142,296.288921,296.264041
2026-05-18 00:28:02,"""89.37.98.34""","""133.69.15.4""",3,3,0,224.810097,224.911966,224.854523


### Traceroute

In [5]:
traceroute_df = pl.scan_parquet(traceroute_paths).filter(
    pl.col("af") == 4,  # limit experiments to only IPv4
    pl.col("lts") < 3600,  # only keep measurements that synced their clock within the last hour
).select([
    "timestamp",
    "from",
    "dst_addr",
    "hop_count",
]).rename({
    "from": "src_addr",
}).unique(
    ["src_addr", "dst_addr"]
).with_columns(
    timestamp=pl.from_epoch(pl.col("timestamp"), time_unit="s")
)

traceroute_df.head().collect()

timestamp,src_addr,dst_addr,hop_count
datetime[μs],str,str,i64
2026-05-18 00:37:41,"""165.165.117.4""","""208.91.191.1""",10
2026-05-18 00:34:11,"""167.179.75.145""","""198.97.190.53""",8
2026-05-18 00:03:04,"""170.210.5.200""","""201.219.153.38""",255
2026-05-18 00:55:48,"""193.174.241.243""","""142.251.142.46""",6
2026-05-18 00:14:45,"""145.239.14.84""","""85.158.212.193""",15


### Cache

In [6]:
ping_df.sink_parquet("ping_df.parquet")
traceroute_df.sink_parquet("traceroute_df.parquet")

## Data Enrichment

### Geo Information

In [7]:
src_ping_addrs = ping_df.select("src_addr").unique().collect()["src_addr"].to_list()
dst_ping_addrs = ping_df.select("dst_addr").unique().collect()["dst_addr"].to_list()
all_unique_addrs = list(set(src_ping_addrs) | set(dst_ping_addrs) )

len(all_unique_addrs)

11109

In [8]:
def chunks(items, size=1024):
    for i in range(0, len(items), size):
        yield items[i:i + size]


client = ipr.IpregistryClient(
    os.getenv("IPREGISTRY_API_KEY"),
    cache=ipr.InMemoryCache(maxsize=50_000, ttl=7 * 24 * 3600),
)

responses: list[ipr.IpInfo] = []
for batch in chunks([ip for ip in all_unique_addrs if ip], 1024):
    print("Requesting batch of size", len(batch))
    result = client.batch_lookup_ips(batch)
    responses.extend(result.data)

print("done")

Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 1024
Requesting batch of size 868
done


In [9]:
rows = []
for resp in responses:
    if type(resp) is ipr.LookupError:
        continue

    rows += [[
        resp.ip,
        resp.location.continent.name,
        resp.location.country.name,
        resp.location.longitude,
        resp.location.latitude,
        resp.connection.asn,
        resp.connection.type,
        resp.connection.domain,
        resp.carrier.mcc is not None,
        resp.security.is_cloud_provider,
        resp.security.is_proxy,
        resp.security.is_vpn,
        resp.security.is_tor,
        resp.security.is_tor_exit,
    ]]

ipinfo_schema = {
    "ip": pl.String,
    "continent": pl.String,
    "country": pl.String,
    "longitude": pl.Float64,
    "latitude": pl.Float64,
    "asn": pl.Int64,
    "conn_type": pl.String,
    "provider": pl.String,
    "is_mobile": pl.Boolean,
    "is_cloud_provider": pl.Boolean,
    "is_proxy": pl.Boolean,
    "is_vpn": pl.Boolean,
    "is_tor": pl.Boolean,
    "is_tor_exit": pl.Boolean,
}

ipinfo_df = pl.DataFrame(
    rows,
    schema=ipinfo_schema,
    orient="row",
)
ipinfo_df.head()

ip,continent,country,longitude,latitude,asn,conn_type,provider,is_mobile,is_cloud_provider,is_proxy,is_vpn,is_tor,is_tor_exit
str,str,str,f64,f64,i64,str,str,bool,bool,bool,bool,bool,bool
"""136.50.32.241""","""North America""","""United States""",-98.49459,29.42519,16591,"""isp""","""googlefiber.net""",false,false,false,false,false,false
"""78.41.115.110""","""Europe""","""Austria""",16.37214,48.20841,35492,"""hosting""","""funkfeuer.at""",false,true,false,false,false,false
"""199.80.10.137""","""North America""","""United States""",-86.30005,32.36672,396440,"""education""","""aum.edu""",false,false,false,false,false,false
"""173.92.58.24""","""North America""","""United States""",-80.60071,35.49146,11426,"""isp""","""charter.net""",false,false,false,false,false,false
"""171.22.151.71""","""Europe""","""Albania""",19.81863,41.32738,209465,"""business""","""vig.al""",false,false,false,false,false,false


### Data Combination

In [10]:
traceroute_df.head().collect()

timestamp,src_addr,dst_addr,hop_count
datetime[μs],str,str,i64
2026-05-18 00:01:16,"""185.206.184.74""","""212.127.78.138""",10
2026-05-18 00:03:55,"""66.244.16.33""","""137.39.1.3""",10
2026-05-18 00:32:46,"""166.0.67.4""","""160.30.224.19""",19
2026-05-18 00:18:46,"""107.162.217.5""","""200.9.157.207""",9
2026-05-18 00:14:21,"""186.55.89.87""","""123.176.1.4""",11


In [22]:
full_df = ping_df.join(
    ipinfo_df.rename({col: f"src_{col}" for col in ipinfo_df.columns}).lazy(),
    left_on="src_addr",
    right_on="src_ip",
    how="left",
).join(
    ipinfo_df.rename({col: f"dst_{col}" for col in ipinfo_df.columns}).lazy(),
    left_on="dst_addr",
    right_on="dst_ip",
    how="left",
).join(
    traceroute_df.drop("timestamp"),
    on=["src_addr", "dst_addr"],
    how="left",
)

In [23]:
full_df = full_df.with_columns(
    # haversine formula: https://en.wikipedia.org/wiki/Haversine_formula
    distance_km=(
        2
        * 6371.0 # earth radius in km
        * (
            (
                (((pl.col("dst_latitude") - pl.col("src_latitude")).radians() / 2).sin() ** 2)
                + pl.col("src_latitude").radians().cos()
                * pl.col("dst_latitude").radians().cos()
                * (((pl.col("dst_longitude") - pl.col("src_longitude")).radians() / 2).sin() ** 2)
            )
            .sqrt()
            .arcsin()
        )
    ),
    src_prefix_8=pl.col("src_addr").str.split(".").list.slice(0, 1).list.join("."),
    src_prefix_16=pl.col("src_addr").str.split(".").list.slice(0, 2).list.join("."),
    dst_prefix_8=pl.col("dst_addr").str.split(".").list.slice(0, 1).list.join("."),
    dst_prefix_16=pl.col("dst_addr").str.split(".").list.slice(0, 2).list.join("."),
)

In [24]:
full_df.select("src_addr").count().collect()

src_addr
u32
1116767


In [30]:
full_df.sink_parquet("2026-05-19-lat-est.parquet")

## Data Checks

In [28]:
# Number of samples
num_samples = full_df.collect().shape[0]

# Number of features
num_features = full_df.collect().shape[1]

# Display these dataset characteristics
print(f"Number of samples: {num_samples}")
print(f"Number of features: {num_features}")

# Display the first few rows of the dataframe to show the structure
print("Example data:")
print(full_df.head().collect())



Number of samples: 1116767
Number of features: 41
Example data:
shape: (5, 41)
┌────────────┬────────────┬────────────┬──────┬───┬────────────┬───────────┬───────────┬───────────┐
│ timestamp  ┆ src_addr   ┆ dst_addr   ┆ sent ┆ … ┆ src_prefix ┆ src_prefi ┆ dst_prefi ┆ dst_prefi │
│ ---        ┆ ---        ┆ ---        ┆ ---  ┆   ┆ _8         ┆ x_16      ┆ x_8       ┆ x_16      │
│ datetime[μ ┆ str        ┆ str        ┆ i64  ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│ s]         ┆            ┆            ┆      ┆   ┆ str        ┆ str       ┆ str       ┆ str       │
╞════════════╪════════════╪════════════╪══════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2026-05-18 ┆ 107.161.22 ┆ 5.188.108. ┆ 3    ┆ … ┆ 107        ┆ 107.161   ┆ 5         ┆ 5.188     │
│ 00:06:38   ┆ .19        ┆ 64         ┆      ┆   ┆            ┆           ┆           ┆           │
│ 2026-05-18 ┆ 81.211.199 ┆ 94.77.252. ┆ 3    ┆ … ┆ 81         ┆ 81.211    ┆ 94        ┆ 94.77     │
│ 00:51:54  